# ComfyUI + LTX 2.3 GGUF on Kaggle
Base: pogscafe (2202 votes) | Lightricks/ComfyUI-LTXVideo (3.9k)

**Run All t? ??u ?u?n cu?i**
- Cell 1: Cài ??t + download models (~15 phút)
- Cell 2: Start ComfyUI + Url dán xu?ng bên du?i + Gist push
- Cell 3: Keep Alive

---

## Cell 1: Install + Download (~15 phút)

In [ ]:
%%time
import os, sys, subprocess, threading, time, json, requests
from datetime import datetime

HOME = '/kaggle/working'
COMFY = f'{HOME}/ComfyUI'
VENV = f'{HOME}/venv'
os.chdir(HOME)

print('=== B1: Environment ===')
!pip install -q virtualenv
if not os.path.exists(VENV):
    !virtualenv {VENV} -p $(which python3.10)
    if not os.path.exists(f'{VENV}/bin/python3.10'):
        !cp /usr/bin/python3.10 {VENV}/bin/
    !ln -sf {VENV}/bin/python3.10 {VENV}/bin/python
    !ln -sf {VENV}/bin/python3.10 {VENV}/bin/python3
print('venv OK')

PIP = f'{VENV}/bin/pip'
PYTHON = f'{VENV}/bin/python'

print('=== B2: Clone ComfyUI + custom nodes ===')
COMFY_COMMIT = '7fc3ccdcc2fb1f20c4b7dd4aca374db952fd66df'
if not os.path.exists(COMFY):
    !git clone https://github.com/comfyanonymous/ComfyUI.git
os.chdir(COMFY)
!git checkout {COMFY_COMMIT} 2>/dev/null
!{PIP} install -q -r requirements.txt

os.chdir(f'{COMFY}/custom_nodes')
for url,name in [
    ('https://github.com/Lightricks/ComfyUI-LTXVideo.git','ComfyUI-LTXVideo'),
    ('https://github.com/logtd/ComfyUI-LTXTricks.git','ComfyUI-LTXTricks'),
    ('https://github.com/city96/ComfyUI-GGUF.git','ComfyUI-GGUF'),
    ('https://github.com/ltdrdata/ComfyUI-Manager.git','ComfyUI-Manager'),
]:
    if not os.path.exists(name):
        !git clone {url}
        print(f'  {name}')
print('ComfyUI + nodes OK')

print('=== B3: Symlink models -> /tmp ===')
!mkdir -p /tmp/models/{unet,clip,vae}
for d in ['unet','clip','vae']:
    src = f'{COMFY}/models/{d}'
    dst = f'/tmp/models/{d}'
    if os.path.islink(src) or os.path.exists(src):
        !rm -rf {src}
    !ln -sf {dst} {src}

print('=== B4: Download LTX-2.3 GGUF (12.4 GB) ===')
os.chdir('/tmp/models/unet')
f = 'LTX-2.3-22B-distilled-1.1-Q2_K.gguf'
if not os.path.exists(f):
    !wget -c 'https://huggingface.co/QuantStack/LTX-2.3-GGUF/resolve/main/LTX-2.3-distilled-1.1/LTX-2.3-22B-distilled-1.1-Q2_K.gguf' -O '{f}'
print(f'Model: {os.path.getsize(f)/1e9:.1f} GB')

print('=== B5: Download text encoder + VAE ===')
for d,f,s in [
    ('clip','gemma-2b.safetensors','text_encoder/model.safetensors'),
    ('vae','ltx-vae.safetensors','vae/vae.safetensors'),
]:
    fp = f'/tmp/models/{d}/{f}'
    if not os.path.exists(fp):
        !wget -c 'https://huggingface.co/Lightricks/LTX-2/resolve/main/{s}' -O '{fp}'
print('All models ready')

print('=== B6: Start ComfyUI ===')
os.chdir(COMFY)
!pkill -f main.py 2>/dev/null
time.sleep(2)
subprocess.Popen([PYTHON,'main.py','--headless','--port','8188','--listen','127.0.0.1','--highvram'],
    stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
for i in range(40):
    time.sleep(3)
    try:
        r = requests.get('http://127.0.0.1:8188/object_info', timeout=2)
        if r.status_code == 200:
            print(f'ComfyUI API ready!')
            break
    except: pass

print('\n=== COMPLETE: Setup done! Go to next cell ===')

## Cell 2: Tunnel Pinggy

In [ ]:
# Tunnel Pinggy (background thread)
TUNNEL_URL = None

def tunnel():
    global TUNNEL_URL
    p = subprocess.Popen(['ssh','-p','443','-R0:localhost:8188','a.pinggy.io'],
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, universal_newlines=True)
    for line in p.stdout:
        if 'https://' in line:
            i = line.find('https://')
            TUNNEL_URL = line[i:].strip().split()[0]
            open(f'{HOME}/url.txt','w').write(TUNNEL_URL)
            print(f'TUNNEL URL: {TUNNEL_URL}')
            break

threading.Thread(target=tunnel, daemon=True).start()

# Wait for URL
for i in range(30):
    time.sleep(5)
    if TUNNEL_URL: break
    try:
        TUNNEL_URL = open(f'{HOME}/url.txt').read().strip()
    except: pass

if TUNNEL_URL:
    print(f'\nURL: {TUNNEL_URL}')
    print('\nCopy the URL above and paste in the next cell!')
else:
    print('No tunnel yet')

## Cell 3: Push URL to Gist

In [ ]:
# Paste URL here:
TUNNEL_URL = ""   # <-- Paste the https://xxx.pinggy.link URL here!

if TUNNEL_URL:
    GIST_ID = "8da27f2e6e0d8809a043712cd90f9237"
    data = {}
    try:
        r = requests.get(f'https://api.github.com/gists/{GIST_ID}', timeout=5)
        if r.status_code == 200:
            c = r.json()['files']['kaggle_backends.json']['content']
            data = json.loads(c) if c.strip() else {}
    except: pass
    data['kaggle-a'] = {
        'url': TUNNEL_URL, 'status': 'online',
        'updated': datetime.now().isoformat(),
        'capabilities': ['t2v','i2v','v2v'], 'gpu': 't4'
    }
    r = requests.patch(f'https://api.github.com/gists/{GIST_ID}',
        json={'files':{'kaggle_backends.json':{'content':json.dumps(data,indent=2)}}}, timeout=10)
    if r.status_code == 200:
        print(f'Gist updated! URL: {TUNNEL_URL}')
        print(f'Hermes can now dispatch')
    else:
        print(f'Gist error: {r.status_code}')
else:
    print('Please paste the URL in the cell above')

## Cell 4: Keep Alive (run in background)

In [ ]:
try:
    while True:
        time.sleep(300)
        if TUNNEL_URL:
            requests.patch(f'https://api.github.com/gists/8da27f2e6e0d8809a043712cd90f9237',
                json={'files':{'kaggle_backends.json':{'content':'{"kaggle-a":{"status":"online"}}'}}},
                timeout=10)
        print('.', end='')
except:
    print('Stopped')